**!!! NOTEBOOK FOR USAGE REFERENCE !!!**

This notebook is not intended to provide actual code; Instead, it is just a "getting started", something you can take as inspiration when using this testbed.

For developers: please do not commit the outputs to the library. Always clear all outputs before saving.

----

Downloads, enrich, filter, save splits

Generates metadata_filtered

General notebook structure:
1. Inicial set: ~300 entities to be partially enriched
2. Enriched set: fully enriched and verified, including manual checks/inputs. Manually remove mismaches name<->pantheon.
3. Filtered set: filtered 100 entities based on representativenss balancing
4. Entities appear exactly in this order in the table representations.
5. Images should only be saved for the entities in the final enriched set


In [ ]:
import os
import sys
import json
import dotenv
import pandas as pd
from PIL import Image
import torch.utils.checkpoint

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 30)

sys.path.append('../TRDP-unlearning')
dotenv.load_dotenv('../TRDP-unlearning/SD_lora_distil/.env')
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0
#!huggingface-cli login --token ${HF_TOKEN}

from vision_unlearning.datasets import pantheon_find_closest_match, download_dataset_pantheon, balanced_subsample_lib
from vision_unlearning.datasets.testbed import calculate_similarity_clip, plot_heatmap
from vision_unlearning.datasets import download_dataset_lfw, count_classes_dataset_lfw
from vision_unlearning.metrics import MetricRace
from vision_unlearning.utils.logger import get_logger, setup_loggers


logger = get_logger('testbed')
setup_loggers()

In [ ]:
dataset_base_path = 'assets/datasets/lfw_splits'
dataset_base_path_filtered = 'assets/datasets/lfw_splits_filtered'

# Restart from step:
#!rm assets/metadata_people_1_enriched_but_not_filtered.json && rm assets/metadata_people_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm assets/metadata_people_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm -r {dataset_base_path_filtered}

#!rm -r {dataset_base_path}

#!rm "assets/similarity_clip_people.json"

In [ ]:
##############################
# Step 1: prepare datasets
##############################
# Prepare unfiltered LFW
sorted_counts = count_classes_dataset_lfw()
for name, n in sorted_counts[:10]:
    print(f"{name}: {n}")

target = "all"  # No class separation, just to get properties for filtering
dataset_forget_name = f"{dataset_base_path}/{target}/train_forget"
dataset_retain_name = f"{dataset_base_path}/{target}/train_retain"

#!rm -r {dataset_base_path}
if not os.path.exists(dataset_base_path):
    download_dataset_lfw(dataset_forget_name, dataset_retain_name, target)

!find {dataset_forget_name} -type f | wc -l
!find {dataset_retain_name} -type f | wc -l
#!du -hs "{dataset_base_path}/{target}"

In [ ]:
df_pantheon = download_dataset_pantheon()

In [ ]:
##############################
# Step 2: attribute inference
##############################
if os.path.exists('assets/metadata_people_1_enriched_but_not_filtered.json'):
    with open(f"assets/metadata_people_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    # Inicial set
    metadata = [{'name': name, 'dataset_n_original': n} for name, n in sorted_counts[:500]]

    # TODO: allow field-by-field inference, so we don't have to recalculate everyone when changing the code?

    # Attribute inference option 1: impossible, no labels in LFW
    
    # Attribute inference option 2: Pantheon
    logger.info('---------------- Enriching metadata with Pantheon')
    for entity in metadata:
        name_pantheon = pantheon_find_closest_match(df_pantheon, entity['name'])
        if name_pantheon is None:
            continue
        entity['name_pantheon'] = name_pantheon
    
        pantheon_row = df_pantheon[df_pantheon['slug']==entity['name_pantheon']].iloc[0]
        entity['birthyear'] = int(pantheon_row['birthyear'])
        entity['gender'] = pantheon_row['gender']
        entity['occupation'] = pantheon_row['occupation'].capitalize()
        entity['bplace_country'] = pantheon_row['bplace_country']
        entity['hpi'] = pantheon_row['hpi']
    
    # Attribute inference option 3: average value in some labeled dataset
    # TODO
    
    # Attribute inference option 4: data-driven with DeepFace
    logger.info('---------------- Enriching metadata with DeepFace')
    metric_race = MetricRace()
    for i, entity in enumerate(metadata):
        if i % 100 == 0:
            print(f"Enriched {i}...")
        # TODO: use multiple images, define race by majority voting
        img = Image.open(f"assets/datasets/lfw_splits/all/train_retain/{entity['name']}_0001.jpg")
        entity['race'] = metric_race.score(img)['race']

    # Save
    with open(f"assets/metadata_people_1_enriched_but_not_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

In [ ]:
##############################
# Step 3: filter
##############################
balanced_n = 100

#!rm assets/metadata_people_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
if os.path.exists('assets/metadata_people_2_enriched_filtered.json'):
    logger.info('Reading existing filtered data')
    with open(f"assets/metadata_people_2_enriched_filtered.json", "r", encoding="utf-8") as f:
        metadata_filtered = json.load(f)
else:
    logger.info('Filtering')
    with open(f"assets/metadata_people_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    df = pd.DataFrame(metadata)
    #df.info()
    
    df['occupation_simplified'] = df['occupation']
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Actor', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Singer', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Musician', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Film director', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Comedian', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Writer', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Artist', 'Artist')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Model', 'Artist')
    
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Tennis player', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Basketball player', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Racing driver', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Swimmer', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Athlete', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Golfer', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Boxer', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Cyclist', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Skater', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Soccer player', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Baseball player ', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('American football player', 'Athlete')
    df['occupation_simplified'] = df['occupation_simplified'].str.replace('Cricketer', 'Athlete')
    df = df[df['occupation_simplified'].isin(['Politician', 'Artist', 'Athlete'])]
    print(df.value_counts('occupation_simplified'))
    
    df["hpi_bin"] = pd.qcut(
        df["hpi"],
        q=[0, 0.25, 0.5, 0.75, 1.0],
        labels=["Q0_25", "Q25_50", "Q50_75", "Q75_100"],
        duplicates="raise"
    )
    
    #df['race'] = df['race'].str.replace('indian', 'indian_middleEastern_latinoHispanic')
    #df['race'] = df['race'].str.replace('middle eastern', 'indian_middleEastern_latinoHispanic')
    #df['race'] = df['race'].str.replace('latino hispanic', 'indian_middleEastern_latinoHispanic')
    print(f'Number of unfiltered rows_ {len(df)}')
    
    df_balanced = balanced_subsample_lib(df, group_cols=['occupation_simplified', 'hpi_bin'], target=balanced_n, priority_col='dataset_n_original')
    
    print(df_balanced.value_counts('occupation_simplified'))
    print('-'*50)
    print(df_balanced.value_counts('hpi_bin'))
    print('-'*50)
    print(df_balanced.value_counts(['occupation_simplified', 'hpi_bin']))
    
    #df[(df['gender']=='F') & (df['race']=='black')]  # There are just 6 black woman, which can't be balanced
    metadata_filtered = df_balanced.to_dict(orient='records')

    with open(f"assets/metadata_people_2_enriched_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata_filtered, f, indent=2)

assert type(metadata_filtered) == list
assert len(metadata_filtered) == balanced_n
assert metadata_filtered[0]['name'] == 'George_W_Bush', 'I think your size-priotity is wrong'

In [ ]:
##############################
# Step 4: save splits
##############################
smallest_entity = min([entity['dataset_n_original'] for entity in metadata_filtered])
restrict_labels = [e['name'] for e in metadata_filtered]
logger.info(f"We have {balanced_n} identities, each one with {smallest_entity} images")

#!rm -r {dataset_base_path_filtered}
if not os.path.exists(dataset_base_path_filtered):
    for i, target in enumerate(restrict_labels):
        logger.info(f"Saving split dataset for entity {i}: {target}")
        dataset_forget_name = f"{dataset_base_path_filtered}/{target}/train_forget"
        dataset_retain_name = f"{dataset_base_path_filtered}/{target}/train_retain"
        class_to_number = download_dataset_lfw(
            dataset_forget_name,
            dataset_retain_name,
            target,
            forget_max_img = smallest_entity,
            retain_max_img_per_class= smallest_entity,
            restrict_labels = restrict_labels,
        )
        assert sum([v>0 for v in class_to_number.values()]) == balanced_n, 'More entities than expected were saved'
        assert sum([v==smallest_entity for v in class_to_number.values()]) == balanced_n, 'Not all entities have the same number of images'
        assert target in class_to_number.keys()
    
        assert sum(len(files) for _, _, files in os.walk(dataset_forget_name)) == smallest_entity+1
        assert sum(len(files) for _, _, files in os.walk(dataset_retain_name)) == (balanced_n-1)*smallest_entity+1

# Each identity has about 4.4Mb
#!du -hs "{dataset_base_path_filtered}/{target}"

#!find {dataset_forget_name} -type f | wc -l
#!find {dataset_retain_name} -type f | wc -l

In [ ]:
!ls assets/datasets/lfw_splits_filtered/George_W_Bush/train_forget

In [ ]:
!ls assets/datasets/lfw_splits_filtered/George_W_Bush/train_retain

In [ ]:
##############################
# Step 5: similarity matrix
##############################
# TODO: move to standalone **Notebook 2: Data Exploration**
#!rm "assets/similarity_clip_people.json"
df_similarities_clip = calculate_similarity_clip('people', restrict_labels)
plot_heatmap(df_similarities_clip)
df_similarities_clip.head()